# Узел Б — обучение модели и Flask

Пошаговая шпаргалка для ручной работы на хосте Б. Обучение и запуск сервера — отдельные команды; Flask не стартует при сборке или запуске контейнера. Сначала выполните общую настройку из корневого README.


## 1. Запустить idle-контейнер

Из корня проекта на хосте Б:

```bash
cd node_b
cp .env.example .env
```

Выполняйте `cp` только при первоначальной настройке, не поверх настроенного файла. Проверьте `TARGET_COLUMN=species`, `TEST_SIZE=0.2`, `RANDOM_STATE=42`, затем:

```bash
docker compose up --build -d
docker ps --filter name=ml-trainer
```

Контейнер простоит без обучения и сервера. Входной `./data` подключён как `/home/DATA:ro`, каталог `./artifacts` — как доступный для записи `/home/MODEL`.


## 2. Проверить данные и обучить

После ручной передачи файла с хоста А проверьте путь и наличие CSV. Если вы сейчас в `node_b`, команды такие:

```bash
realpath data
ls -l data/clean_data.csv
```

Если вы в корне проекта, используйте `realpath node_b/data` и `ls -l node_b/data/clean_data.csv`. Запустите обучение:

```bash
docker exec ml-trainer python /app/train_model.py
```

Скрипт стратифицированно делит данные на 80/20, обучает pipeline только на train-части, сохраняет `/home/MODEL/model.joblib` (pipeline и метаданные) и `/home/MODEL/test_data.csv` (признаки и истинный класс), затем завершается. Для Iris это 120 строк обучения и 30 тестовых. Новое обучение выполняется только повторным ручным запуском этой команды.


## 3. Запустить Flask в foreground

После завершения обучения запустите сервер отдельной командой:

```bash
docker exec -it ml-trainer python /app/ml_service.py
```

Терминал занят, пока Flask слушает `0.0.0.0:5000`. Для остановки нажмите `Ctrl+C`; контейнер останется запущенным и простаивающим. `/predict` выполняет inference только на сохранённых тестовых строках, не обучает модель и не делит данные повторно. При отсутствующих артефактах endpoint возвращает `503`.


## 4. Проверить состояние и результат

Пока сервер работает, на хосте А проверьте:

```bash
docker exec data-processor curl http://IP_B:5000/health
docker exec data-processor curl http://IP_B:5000/predict
```

`/health` возвращает `status: "ok"`, `model_ready` и `test_data_ready`. Успешный `/predict` возвращает `status: "success"`, accuracy, 120 train-строк, 30 test-строк и 30 объектов `predictions` с полями `row_index` и `prediction`. `row_index` — индекс строки очищенного CSV, начиная с 0. Если HTTP недоступен, проверьте IP хоста Б, сеть, firewall и то, что Flask всё ещё работает.


## 5. Firewall и отчёт

На хосте Б проверьте текущие правила до изменения:

```bash
sudo iptables -t nat -nvL
sudo iptables -nvL DOCKER-USER --line-numbers
```

Настройте правило `ACCEPT` для IP хоста А на TCP 5000 первым, затем `DROP` для остальных на TCP 5000; сохраните правила через `sudo netfilter-persistent save`. Не добавляйте дубликаты без проверки порядка. `ping` проверяет ICMP, не Flask-порт.

В отчёте студент вручную описывает скрипты и добавляет требуемые преподавателем скриншоты подготовки файла, обучения, `/health` и `/predict`. Проверка двух физических хостов выполняется на самих хостах.
